# Indices de lexicalité

In [ ]:
pip install wordfreq psycopg2-binary

In [ ]:
import re
import unicodedata
from wordfreq import zipf_frequency
#import psycopg2
#import sqlite3
from multiprocessing import Pool, cpu_count
import csv
import duckdb

In [ ]:
# Extrait uniquement les "mots alphabétiques"
def tokenize(text):
    return re.findall(r"\b[a-zA-Z]+\b", text)

In [ ]:
# Calcule lexicalité et stats à partir d’un texte
def analyze_text(text, threshold=2.5, lang="en"):
    normalized = normalize_text(text)
    tokens = tokenize(normalized)

    total_words = len(tokens)

    valid_words = []
    invalid_words = []

    for word in tokens:
        freq = zipf_frequency(word, lang)
        if freq >= threshold:
            valid_words.append(freq)
        else:
            invalid_words.append(word)

    valid_count = len(valid_words)
    invalid_count = len(invalid_words)

    lexicality = (valid_count / total_words * 100) if total_words else 0
    avg_zipf = sum(valid_words) / len(valid_words) if valid_words else 0

    return {
        "total_words": total_words,
        "valid_words": valid_count,
        "invalid_words": invalid_count,
        "lexicality": lexicality,
        "avg_zipf": avg_zipf,
    }

In [ ]:
# Lit la base par blocs 
def stream_data(batch_size=1000):
    con = duckdb.connect("../data/ocr.db")
    offset = 0
    while True:
        batch = con.sql(f"""
            SELECT id, decode(content) AS content_text
            FROM fulltext_ocr
            LIMIT {batch_size} OFFSET {offset}
        """).fetchall()
        if not batch:
            break
        yield batch
        offset += batch_size

In [ ]:
# Décode le blob et analyse un document
def process_row(row, threshold=2.5, lang="en"):
    doc_id, content = row

    if content is None:
        return None

    # BLOB → bytes
    if isinstance(content, bytes):
        try:
            text = content.decode("utf-8", errors="ignore")
        except:
            # fallback si encodage exotique
            text = content.decode("latin-1", errors="ignore")
    else:
        text = str(content)

    analysis = analyze_text(text, threshold, lang)

    return {
        "id": doc_id,
        **analysis
    }

In [ ]:
# Applique multiprocessing sur les batches de données
def process_database_parallel(batch_size=1000, threshold=2.5, lang="en"):
    n_workers = max(cpu_count() - 1, 1)
    print(f"Workers utilisés: {n_workers}")

    results = []

    with Pool(n_workers) as pool:
        for batch in stream_data(batch_size):

            processed = pool.starmap(
                process_row,
                [(row, threshold, lang) for row in batch]
            )

            results.extend([r for r in processed if r is not None])

    return results

In [ ]:
# Agrège les résultats globaux
def compute_global_summary(report):
    total_docs = len(report)
    total_words = sum(r["total_words"] for r in report)
    total_valid = sum(r["valid_words"] for r in report)

    lexicality = (total_valid / total_words * 100) if total_words else 0

    return {
        "total_docs": total_docs,
        "total_words": total_words,
        "lexicality": lexicality
    }

In [ ]:
# Sauvegarde 
def export_report_to_csv(report, output_path="report.csv"):
    if not report:
        return

    fieldnames = report[0].keys()

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(report)

In [ ]:
# Lance tout le pipeline
BATCH_SIZE = 1000
THRESHOLD = 2.5
LANG = "en"

report = process_database_parallel(
    batch_size=BATCH_SIZE,
    threshold=THRESHOLD,
    lang=LANG
)

summary = compute_global_summary(report)

print("\n=== GLOBAL SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v}")

export_report_to_csv(report, "ocr_lexicality-report.csv")